# Phase 5 — Feature Engineering

This phase converts product titles into consistent categories and product specifications. The source file remains unchanged. The shared parser in `backend/product_features.py` is used here and by the live application, so training and live inference use the same category rules and title extraction.

In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from backend.product_features import classify_product, extract_title_specs, product_feature_frame

DATA_PATH = Path("../electronics_analysis_ready.csv")
OUTPUT_PATH = Path("electronics_feature_engineered.csv")
PHASE_OUTPUT_DIR = Path("phase_outputs")
PHASE_OUTPUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH).copy()
print("Source shape:", df.shape)
print("Source rows retained:", len(df))
print("Source columns:", df.columns.tolist())

Source shape: (5010, 17)
Source rows retained: 5010
Source columns: ['name', 'main_category', 'sub_category', 'image', 'link', 'no_of_ratings', 'discount_price', 'actual_price', 'review_rating', 'review_text', 'discount_price_missing', 'actual_price_missing', 'review_count_missing', 'review_text_missing', 'price_available', 'discount_amount', 'discount_percent']


## Product categories

Rules inspect accessory identity before compatible device brands. A phone title beginning with a known model stays a smartphone even if the listing mentions an included charger. Headphones are recognized before phone-maker prefixes, which prevents titles such as OnePlus Nord Buds from becoming smartphones. Unclear products remain explicitly unclassified.

In [4]:
df["product_category"] = df["name"].map(classify_product)
category_counts = df["product_category"].value_counts().rename_axis("product_category").to_frame("count")
display(category_counts)
print("Total classified:", int((df.product_category != "Other / Unclassified").sum()))
print("Unclassified:", int((df.product_category == "Other / Unclassified").sum()))
print("Non-electronic / miscellaneous:", int((df.product_category == "Non-electronic / Miscellaneous").sum()))

unclassified = df.loc[df.product_category.eq("Other / Unclassified"), ["name"]]
print("Representative unclassified titles (first 30, source order):")
display(unclassified.head(30))

                                count
product_category                     
Other / Unclassified             1050
Headphones and Earphones          598
Mobile Accessories                532
Chargers and Cables               487
Smartphones                       345
Computer Accessories              330
Smartwatches                      214
Laptops                           188
Batteries                         186
Tablets                           150
Storage Devices                   130
Non-electronic / Miscellaneous    120
Speakers                           99
Remote Controls                    91
Printers and Printing Supplies     84
Camera Accessories                 60
Power Banks                        55
Televisions                        51
Networking Devices                 47
Monitors and Projectors            46
Cameras                            33
Smartwatch Accessories             30
Electrical Accessories             30
Telephones                         19
Computer Com

## Extract specifications and review representative records

The six values are title-derived features, so a blank value means the title did not provide enough evidence. Category gates prevent screen sizes and refresh rates from being extracted from unrelated products. These checks cover common title patterns and confirm that accessories and compatible-device mentions do not override the product identity.

In [6]:
spec_columns = ["ram_gb", "storage_gb", "battery_mah", "screen_inches", "charging_w", "refresh_rate_hz"]
specs = pd.DataFrame(
    [extract_title_specs(row.name, row.product_category) for row in df.itertuples()],
    index=df.index,
)
# itertuples exposes the title in the `name` field; make the assignment explicit.
df[spec_columns] = specs[spec_columns]

# Curated examples drawn from the source data and inspected for title meaning.
checks = [
    (r"^redmi 10 power", "Smartphones", "ram_gb", 8.0),
    (r"^oneplus nord ce 2 lite", "Smartphones", "storage_gb", 128.0),
    (r"^oneplus bullets z2", "Headphones and Earphones", None, None),
    (r"^mi lithium ion xiaomi wireless power bank", "Power Banks", "battery_mah", 10000.0),
    (r"^hp 14s, 5th gen", "Laptops", "storage_gb", 512.0),
    (r"^apple 20w usb-c power adapter", "Chargers and Cables", "charging_w", 20.0),
    (r"^spigen tough armor back cover", "Mobile Accessories", None, None),
    (r"^samsung galaxy m53 5g", "Smartphones", "ram_gb", 6.0),
    (r"^iphone original 20w charger", "Chargers and Cables", "charging_w", 20.0),
    (r"^gizga essentials universal silicone keyboard protector", "Computer Accessories", "screen_inches", np.nan),
]
check_rows = []
for pattern, expected_category, feature, expected_value in checks:
    match = df[df.name.str.contains(pattern, case=False, regex=True, na=False)]
    if match.empty:
        check_rows.append({"pattern": pattern, "found": False, "expected_category": expected_category, "actual_category": None, "feature": feature, "expected": expected_value, "actual": None, "pass": False})
        continue
    row = match.iloc[0]
    actual_value = row[feature] if feature else None
    value_ok = feature is None or (pd.isna(expected_value) and pd.isna(actual_value)) or (pd.notna(actual_value) and pd.notna(expected_value) and np.isclose(actual_value, expected_value))
    ok = row.product_category == expected_category and value_ok
    check_rows.append({"pattern": pattern, "found": True, "expected_category": expected_category, "actual_category": row.product_category, "feature": feature, "expected": expected_value, "actual": actual_value, "pass": bool(ok)})
checks_df = pd.DataFrame(check_rows)
display(checks_df)
assert checks_df["pass"].all(), "Review the specific category/spec extraction examples before proceeding."

coverage = (df[spec_columns].notna().mean() * 100).rename("coverage_percent").round(2).to_frame()
display(coverage)
print("Mean coverage across six specification features:", round(float(coverage.coverage_percent.mean()), 2), "%")
print("Missing values are retained; no records are removed.")

                                                   pattern  found         expected_category           actual_category        feature  expected   actual  pass
0                                          ^redmi 10 power   True               Smartphones               Smartphones         ram_gb       8.0      8.0  True
1                                  ^oneplus nord ce 2 lite   True               Smartphones               Smartphones     storage_gb     128.0    128.0  True
2                                      ^oneplus bullets z2   True  Headphones and Earphones  Headphones and Earphones           None       NaN      NaN  True
3               ^mi lithium ion xiaomi wireless power bank   True               Power Banks               Power Banks    battery_mah   10000.0  10000.0  True
4                                         ^hp 14s, 5th gen   True                   Laptops                   Laptops     storage_gb     512.0    512.0  True
5                           ^apple 20w usb-c power a

In [7]:
# Validate plausibility ranges and inspect a fixed sample of extracted rows.
ranges = {
    "ram_gb": (1, 128), "storage_gb": (1, 8192), "battery_mah": (100, 100000),
    "screen_inches": (1, 100), "charging_w": (1, 300), "refresh_rate_hz": (30, 360),
}
validation = []
for col, (lo, hi) in ranges.items():
    values = df[col].dropna()
    invalid = int((~values.between(lo, hi)).sum())
    validation.append({"feature": col, "available": int(values.size), "invalid_out_of_range": invalid})
validation_df = pd.DataFrame(validation)
display(validation_df)
assert validation_df.invalid_out_of_range.sum() == 0

category_coverage = df.groupby("product_category")[spec_columns].apply(lambda x: x.notna().mean() * 100).round(1)
plt.figure(figsize=(12, 7))
plt.imshow(category_coverage, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
plt.colorbar(label="Coverage (%)")
plt.xticks(range(len(spec_columns)), spec_columns, rotation=35, ha="right")
plt.yticks(range(len(category_coverage)), category_coverage.index)
plt.title("Specification Coverage by Inferred Product Category")
plt.tight_layout()
plt.savefig(PHASE_OUTPUT_DIR / "phase5_feature_coverage.png", dpi=180, bbox_inches="tight")
plt.show()

display(df.loc[df[spec_columns].notna().any(axis=1), ["name", "product_category", *spec_columns]].sample(20, random_state=42))

           feature  available  invalid_out_of_range
0           ram_gb        243                     0
1       storage_gb        263                     0
2      battery_mah        148                     0
3    screen_inches        179                     0
4       charging_w        220                     0
5  refresh_rate_hz         40                     0
                                                                                                                               name     product_category  ram_gb  storage_gb  battery_mah  screen_inches  charging_w  refresh_rate_hz
1531                               tecno spark 8 pro (turquoise cyan, 4gb ram,64gb storage) | 48mp triple camera | 33w fast charger          Smartphones     4.0        64.0          NaN            NaN        33.0              NaN
4728  lenovo v15 amd ryzen 5 5500u 15.6" (39.62cm) fhd 250 nits antiglare thin and light laptop (8gb/512gb ssd/windows 11 home/i...              Laptops     NaN       512.0    

## Phase 5 result

The exported file keeps every source record, original field, category label, and six nullable numerical specifications. Coverage is expected to vary by category and title-writing style. This is ready as an exploratory feature-engineering output; Phase 6 will use imputation inside the training pipeline and will compare whether the sparse specification features improve the target proxy.

In [9]:
df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
print("Final shape:", df.shape)
print("Original rows retained:", len(df))
print("Feature columns:", spec_columns)

Saved: electronics_feature_engineered.csv
Final shape: (5010, 24)
Original rows retained: 5010
Feature columns: ['ram_gb', 'storage_gb', 'battery_mah', 'screen_inches', 'charging_w', 'refresh_rate_hz']
